# Getting Started

This notebook is the on-ramp to the xQTL protocol — a reproducible pipeline for molecular QTL analysis from raw genotypes and phenotypes through discovery, fine-mapping, and integration with GWAS.

A minimal toy dataset of **49 de-identified samples** is used throughout the examples on this site so you can try every pipeline end-to-end before running on real data.

This page walks you from a clean machine to your first successful run in about an hour. If you already have pixi and SoS installed, jump to [Analysis](#analysis).

---

## Before You Start

You'll need a Linux or macOS shell. Windows users: install [WSL2](https://learn.microsoft.com/windows/wsl/install) first.

| Requirement | Minimum | Recommended |
|---|---|---|
| Disk space | 10 GB (minimal install) | 40 GB (full bioinformatics stack) |
| Memory | 16 GB | 50 GB+ on HPC for the installer |
| Network | GitHub, conda-forge, synapse.org | Same |
| Git | Any recent version | 2.30+ |

:::{tip}
**On HPC, start on a compute node.** The installer is memory-hungry and login nodes will kill it. Grab an interactive session first:

```bash
srun --mem=50G --pty bash          # SLURM
bsub -Is -M 50000 -n 4 bash        # LSF
```
:::

---

## Step 1. Install pixi

We manage every dependency — Python, R, JupyterLab, bioinformatics tools — with [pixi](https://pixi.sh/). One installer sets it all up.

```bash
curl -fsSL https://raw.githubusercontent.com/StatFunGen/pixi-setup/refs/heads/main/pixi-setup.sh -o pixi-setup.sh
bash pixi-setup.sh
```

The installer will prompt you for two things:

**1. Installation path** — where pixi stores environments and the package cache.

| Setting | When to use |
|---|---|
| `$HOME/.pixi` (default) | Laptops and workstations with plenty of home-directory space |
| `/lab/$USER/.pixi` or scratch | HPC systems with strict home-directory quotas |

**2. Installation type** — pick based on what you plan to do.

| Type | Size | Files | Includes |
|---|---|---|---|
| **1. minimal** | ~5 GB | ~100k | CLI tools, Python data-science stack, JupyterLab, base R |
| **2. full** | ~35 GB | ~350k | Everything above, **plus** samtools, bcftools, plink2, GATK4, STAR, Seurat, Bioconductor |

Choose **minimal** for xQTL runs with pre-processed inputs; choose **full** if you'll also do upstream QC, alignment, or single-cell work.

**Activate and verify:**

```bash
source ~/.bashrc          # or ~/.zshrc on macOS
pixi --version
```

---

## Step 2. Add SoS

The protocol's pipelines are written as [SoS (Script of Scripts)](https://vatlab.github.io/sos-docs/) workflows. Install the SoS suite into pixi's Python environment:

```bash
pixi global install --environment python -c conda-forge \
    sos sos-pbs sos-notebook jupyterlab-sos \
    sos-bash sos-python sos-r

pixi run -e python python -m sos_notebook.install
```

**Verify:**

```bash
sos --version
jupyter kernelspec list    # should include 'sos'
```

---

## Step 3. Clone the Protocol

```bash
git clone https://github.com/StatFunGen/xqtl-protocol.git
cd xqtl-protocol
```

:::{note}
**What's in the repo**

| Folder | Contents |
|---|---|
| `pipeline/` | The SoS workflows you'll run |
| `code/` | Notebook documentation (this page lives here) |
| `data/` | Small example inputs and configuration templates |
| `website/` | JupyterBook sources for the docs site |
:::


## Analysis

Please visit [the homepage of the protocol website](https://statfungen.github.io/xqtl-protocol/) for general background on this resource, in particular the [How to use the resource](https://statfungen.github.io/xqtl-protocol/README.html#how-to-use-the-resource) section. To perform a complete analysis from molecular phenotype quantification to xQTL discovery, conduct your analysis in the order listed below. Each link contains a mini-protocol for a specific task, and all commands documented in each mini-protocol should be executed from the command line.

### Molecular Phenotype Quantification

Molecular phenotype data is required for the generation of QTLs. We support bulk RNA-seq, methylation, and splicing phenotypes. Multiple [reference data](https://statfungen.github.io/xqtl-protocol/code/reference_data/reference_data.html#) files are required before molecular phenotypes are quantified — reference genomes, gene annotations, variant annotations, linkage disequilibrium data, and topologically associated domains.

- [Quantification of gene expression](https://statfungen.github.io/xqtl-protocol/code/molecular_phenotypes/bulk_expression.html) — RNA-SeQC for gene-level counts, or RSEM for transcript-level counts
- [Quantification of alternative splicing](https://statfungen.github.io/xqtl-protocol/code/molecular_phenotypes/splicing.html) — leafcutter2 to identify alternatively excised introns
- [Quantification of DNA methylation](https://statfungen.github.io/xqtl-protocol/code/molecular_phenotypes/methylation.html) — SeSAMe

Each phenotype then undergoes phenotype-specific quality control and normalization.

### Data Pre-Processing

- [Genotype preprocessing](https://statfungen.github.io/xqtl-protocol/code/data_preprocessing/genotype_preprocessing.html) — variant filters with bcftools, conversion to plink format, kinship and PCA on unrelated individuals
- [Phenotype preprocessing](https://statfungen.github.io/xqtl-protocol/code/data_preprocessing/phenotype_preprocessing.html) — feature annotation, imputation of missing entries, formatting for QTL analysis
- [Covariate preprocessing](https://statfungen.github.io/xqtl-protocol/code/data_preprocessing/covariate_preprocessing.html) — merge phenotypic data with genetic PCs, then compute hidden factors to use as additional covariates

### QTL Association Analysis

- [QTL association analysis](https://statfungen.github.io/xqtl-protocol/code/association_scan/qtl_association_testing.html) — TensorQTL with options for cis, trans, and interaction terms
- [Hierarchical multiple testing](https://statfungen.github.io/xqtl-protocol/code/association_scan/qtl_association_postprocessing.html) — adjust p-values across levels

### Integrative Analysis

- [TWAS](https://statfungen.github.io/xqtl-protocol/code/pecotmr_integration/twas_ctwas.html) — identify genes associated with complex traits
- [Univariate fine-mapping and TWAS with SuSiE](https://statfungen.github.io/xqtl-protocol/code/mnm_analysis/univariate_fine_mapping_twas_vignette.html) — TWAS weights and credible sets
- [Regression with summary statistics](https://statfungen.github.io/xqtl-protocol/code/mnm_analysis/summary_stats_finemapping_vignette.html) — include GWAS summary stats in SuSiE fine-mapping
- [Univariate fine-mapping of functional data](https://statfungen.github.io/xqtl-protocol/code/mnm_analysis/univariate_fine_mapping_fsusie_vignette.html) — fSuSiE with epigenomic annotations
- [Colocalization analysis](https://statfungen.github.io/xqtl-protocol/code/pecotmr_integration/SuSiE_enloc.html) — pairwise colocalization of xQTL and GWAS fine-mapping results
- [Colocboost](https://statfungen.github.io/xqtl-protocol/code/mnm_analysis/mnm_methods/colocboost.html) — alternative shared-variant discovery across multiple molecular traits
- [Excess-of-overlap enrichment](https://statfungen.github.io/xqtl-protocol/code/enrichment/eoo_enrichment.html) — enrichment of significant variants within genomic annotations
- [Pathway enrichment (GSEA)](https://statfungen.github.io/xqtl-protocol/code/enrichment/gsea.html) — overrepresented biological pathways in a gene set
- [Stratified LD Score Regression](https://statfungen.github.io/xqtl-protocol/code/enrichment/sldsc_enrichment.html) — heritability partitioning by annotation


## Data

For record-keeping, preparation of the demo dataset is documented [on this page](https://github.com/cumc/fungen-xqtl-analysis/tree/main/analysis/Wang_Columbia/ROSMAP/MWE) — a private repository accessible to FunGen-xQTL analysis working group members.

All required input data for the protocols on this site live on [Synapse](https://www.synapse.org/#!Synapse:syn36416601). To download it:

1. Create a free account on [synapse.org](https://www.synapse.org/) — username and password are required to download.
2. Install the Synapse API client. It's already packaged for pixi:
   ```bash
   pixi global install -c conda-forge --environment python synapseclient
   ```
   Alternatively, `pip install synapseclient`. See [the Synapse install docs](https://help.synapse.org/docs/Installing-Synapse-API-Clients.1985249668.html) for details.
3. Every folder at each level of the Synapse project has its own unique ID, so you can download just the subset you need.

To download the test data for **Bulk RNA-seq molecular phenotype quantification**:

```python
import synapseclient
import synapseutils
syn = synapseclient.Synapse()
syn.login("your username on synapse.org", "your password on synapse.org")
files = synapseutils.syncFromSynapse(syn, 'syn53174239', path="./")
```

To download the test data for **xQTL association analysis**:

```python
import synapseclient
import synapseutils
syn = synapseclient.Synapse()
syn.login("your username on synapse.org", "your password on synapse.org")
files = synapseutils.syncFromSynapse(syn, 'syn52369482', path="./")
```


## Software Environment

Every protocol on this site is designed to run inside the [pixi](https://pixi.sh/) environment set up in Steps 1–2 above. Once pixi and SoS are installed, every example command "just works" — no per-pipeline container, no manual dependency wrangling.

If you need to add extra software later, install it into the appropriate pixi environment:

```bash
# Python package (into the shared python env)
pixi global install -c conda-forge --environment python <package>

# R package (into the r-base env)
pixi global install -c conda-forge --environment r-base r-<package>

# Standalone CLI tools
pixi global install -c bioconda <tool>
```

### Troubleshooting

**R library conflicts** — if you see errors like

```
Error in dyn.load(file, DLLpath = DLLPath, ...):
unable to load shared object '$PATH/R/x86_64-pc-linux-gnu-library/4.2/stringi/libs/stringi.so':
libicui18n.so.63: cannot open shared object file: No such file or directory
```

your system R libraries are being picked up alongside the pixi ones. Unset them before running the pipeline:

```bash
export R_LIBS=""
export R_LIBS_USER=""
```

**`pixi: command not found`** — open a new terminal or `source ~/.bashrc` (Linux/HPC) / `source ~/.zshrc` (macOS).

**Installer killed on HPC** — you're on a login node. Request a compute node with at least 50 GB of memory and re-run.

**`sos: command not found`** — Step 2 didn't complete. Re-run the `pixi global install` command for SoS.

**`ModuleNotFoundError` during a pipeline** — install the missing package into pixi's python env with the command above.

Still stuck? [Open an issue](https://github.com/StatFunGen/xqtl-protocol/issues) with the command you ran and the full error output.


## Analyses on High Performance Computing Clusters

The protocol example on this page runs on a desktop workstation as a demonstration. Typical production analyses should run on an HPC cluster. SoS supports this natively via [SoS Remote Tasks](https://vatlab.github.io/sos-docs/doc/user_guide/task_statement.html) on [configured host computers](https://vatlab.github.io/sos-docs/doc/user_guide/host_setup.html).

We provide a [toy example for running SoS pipelines on a typical HPC cluster environment](https://github.com/statfungen/xqtl-protocol/blob/main/code/misc/Job_Example.ipynb) — first-time users are encouraged to work through it before launching real jobs, as it covers the host and task configuration you'll reuse for every subsequent pipeline.
